# Lesson 19: Flow loss and Euler sampling

This lesson uses a visual flow experiment and small code cells to show the real SmolVLA path. Run cells in order. The learner notebook has exercises for you to complete.

**Result:** a correct masked flow loss and a sampler that moves from noise to an action chunk.

**Notebook pair:** [Reference answers](19-smolvla-flow-reference.ipynb).

**Evidence boundary:** CPU checks show tensor mechanics. They do not show a useful robot policy. Cells with an experiment tag load local model or LIBERO assets and can take much longer.

In [ ]:
import os
import sys
from pathlib import Path


def find_workspace() -> Path:
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the tutorial workspace.")


WORKSPACE = find_workspace()
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))
CONFIGS = WORKSPACE / "configs"
DATA = WORKSPACE / ".local-data"

In [ ]:
from collections.abc import Callable

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display
from torch.utils.data import default_collate

from vla_lab.teaching.act_model import frame_offset_seconds, open_libero_dataset
from vla_lab.teaching.smolvla import SmolVLAAdapter, flow_training_pair, load_prepared_smolvla

## Model map: flow velocity turns noise into actions

SmolVLA predicts a **velocity** that changes a noisy action chunk into a clean action chunk. For one training example:

$$x_t=(1-t)\,a+t\,\varepsilon, \qquad u_t=\frac{dx_t}{dt}=\varepsilon-a$$

Where:
* $a$ - the clean action chunk.
* $\varepsilon\sim\mathcal{N}(0,I)$ - Gaussian noise with the shape of $a$.
* $t\in[0,1]$ - unitless time: $t=0$ is the clean chunk and $t=1$ is noise.
* $x_t$ - the noisy chunk at time $t$.
* $u_t$ - the target velocity.

During training, each batch row has its own sampled time. A time tensor with shape $(B,)$ is reshaped to $(B,1,1)$ so it broadcasts across that row's action chunk.

$a$, $\varepsilon$, $x_t$, and $u_t$ have shape $(B,T,D_{\max})$. The first $D_{\text{action}}=7$ columns are the normalized robot command channels; the remaining 25 columns are zero padding for a shared model shape. The loss uses only unpadded steps and the first 7 columns:

$$L=\frac{\sum_{b=1}^{B}\sum_{j=1}^{T}\sum_{d=1}^{D_{\text{action}}}v_{b,j}\left(\hat{v}_{b,j,d}-u_{b,j,d}\right)^2}{\max\!\left(N_{\text{valid}}\,D_{\text{action}},\ 1\right)}, \qquad N_{\text{valid}}=\sum_{b,j}v_{b,j}$$

Where:
* $B$ - batch size; $T$ - number of actions in a chunk; $D_{\max}=32$ - padded action width.
* $b$, $j$, $d$ - indices of the batch item, action step, and action column.
* $v_{b,j}$ - 1 for a recorded action, 0 when `action_is_pad[b, j]` is `True`.
* $\hat{v}$ - the predicted velocity; $u$ - the target velocity.
* $N_{\text{valid}}$ - the number of unpadded $(b,j)$ pairs in the batch.

This avoids counting padded actions or padded channels as training targets.

The condition `c` tells the model which action to produce. In SmolVLA it is not one scalar: it includes the encoded camera images, task text, and robot state. The vision-language model encodes this context; the action expert uses it to predict the velocity at each time. The toy widget below reduces this to a scalar target `c` so you can see how changing a condition changes the path. It uses no checkpoint, training, or simulator.

Sampling starts from Gaussian noise at $t=1$ and takes $n$ negative Euler steps toward $t=0$:

$$x\leftarrow x+\Delta t\;v_\theta(x,t,c), \qquad \Delta t=-\frac{1}{n}, \qquad t\in\left\{1,\ 1-\tfrac{1}{n},\ \dots,\ \tfrac{1}{n}\right\}$$

Where:
* $x$ - the current action chunk, initialized to the noise.
* $n$ - the number of Euler steps.
* $v_\theta(x,t,c)$ - the model's velocity at the current $x$ and $t$, given condition $c$.

The output is the predicted action chunk at $t\approx0$.

For a worked scalar example, let $a=2$, $\varepsilon=-1$, and $t=0.25$. Then $x_t=1.25$ and $u=-3$. With three Euler steps, $\Delta t=-1/3$; each update adds $-\tfrac{1}{3}\cdot(-3)=+1$, so integration from $-1$ returns $2$. This uses unitless normalized action values.

### Interactive experiment: change time, noise, and condition

The curve shows a scalar linear flow from a starting noise value $\varepsilon$ at $t=1$ to a condition-selected target $c$ at $t=0$. The marker shows $x_t$ at the selected time. The dashed points show Euler steps. The constant toy velocity is $\varepsilon-c$, so this toy reaches $c$ exactly. Change the condition to see why a model must use the image, language, and state context when it predicts a direction. Change the step count to see the integration grid. This is a numerical picture only; the sliders do not train or call SmolVLA.

In [ ]:
def plot_toy_flow(
    condition: float, initial_noise: float, time_value: float, integration_steps: int
) -> None:
    times = np.linspace(0.0, 1.0, 101)
    path = (1.0 - times) * condition + times * initial_noise
    dt = -1.0 / integration_steps
    euler_times = [1.0]
    euler_values = [initial_noise]
    value = initial_noise
    for index in range(integration_steps):
        time_now = 1.0 + index * dt
        velocity = initial_noise - condition
        value += dt * velocity
        euler_times.append(time_now + dt)
        euler_values.append(value)
    x_now = (1.0 - time_value) * condition + time_value * initial_noise
    figure, axis = plt.subplots(figsize=(7, 3.5))
    axis.plot(times, path, label="linear path $x_t$")
    axis.plot(euler_times, euler_values, "o--", label="Euler samples")
    axis.scatter([time_value], [x_now], color="crimson", zorder=3, label="selected time")
    axis.scatter([0.0, 1.0], [condition, initial_noise], color="black", zorder=3)
    axis.set(xlabel="time t (noise 1 → action 0)", ylabel="scalar action value")
    axis.set_title(
        f"condition c={condition:.1f}; noise ε={initial_noise:.1f}; "
        f"velocity ε−c={initial_noise - condition:.1f}"
    )
    axis.grid(True, alpha=0.25)
    axis.legend()
    figure.tight_layout()
    plt.show()


FLOW_CONDITION = widgets.FloatSlider(
    value=2.0, min=-2.0, max=2.0, step=0.1, description="condition c"
)
FLOW_NOISE = widgets.FloatSlider(value=-1.0, min=-2.0, max=2.0, step=0.1, description="noise ε")
FLOW_TIME = widgets.FloatSlider(value=0.25, min=0.0, max=1.0, step=0.01, description="time t")
FLOW_STEPS = widgets.IntSlider(value=3, min=1, max=12, step=1, description="Euler steps")
FLOW_PLOT = widgets.interactive_output(
    plot_toy_flow,
    {
        "condition": FLOW_CONDITION,
        "initial_noise": FLOW_NOISE,
        "time_value": FLOW_TIME,
        "integration_steps": FLOW_STEPS,
    },
)
display(widgets.VBox([FLOW_CONDITION, FLOW_NOISE, FLOW_TIME, FLOW_STEPS]), FLOW_PLOT)

## Exercise 19.1 — Implement the masked objective and sampler

**Purpose.** Implement two operations in the real training and rollout path.

1. `masked_flow_mse(prediction, target, is_pad, action_dim)` receives prediction and target $(B,T,D_{\max})$, padding mask $(B,T)$ where `True` means ignore, and the real action width $D_{\text{action}}$. Return the loss $L$ from the model map as a scalar tensor. If all steps are padding, return zero without dividing by zero.
2. `euler_to_action(velocity_fn, noise, num_steps)` starts from noise $(B,T,D_{\max})$ and applies the Euler update from the model map with $n$ = `num_steps`. `velocity_fn(x, t)` takes the current chunk and a `float32` time tensor of shape $(B,)$ filled with the current $t$, and returns the velocity with the shape of `x`. Return the final tensor. Reject `num_steps < 1`.

**Numeric check.** For $a=2$, $\varepsilon=-1$, $t=0.25$, the path point is $1.25$ and the target velocity is $-3$. A perfect scalar velocity of $-3$, integrated from $-1$ with three steps, returns $2$.

More behaviour checks for this exercise: `check_masked_flow_mse` and `check_euler_to_action` in [lesson19_checks.py](lesson19_checks.py).

In [ ]:
def masked_flow_mse(
    prediction: torch.Tensor, target: torch.Tensor, is_pad: torch.Tensor, action_dim: int
) -> torch.Tensor:
    # TODO: ignore padded time steps and padded action columns.
    raise NotImplementedError("Implement the masked mean")


def euler_to_action(
    velocity_fn: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    noise: torch.Tensor,
    num_steps: int,
) -> torch.Tensor:
    # TODO: integrate from t=1 toward t=0 with a negative Euler step.
    raise NotImplementedError("Implement Euler sampling")


prediction = torch.tensor([[[1.0, 100.0], [3.0, 100.0]]])
target = torch.zeros_like(prediction)
pad = torch.tensor([[False, True]])
assert masked_flow_mse(prediction, target, pad, action_dim=1).item() == 1.0


def constant_velocity(x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    return torch.full_like(x, -3.0)


assert torch.allclose(
    euler_to_action(constant_velocity, torch.tensor([[[-1.0]]]), 3), torch.tensor([[[2.0]]])
)

The tagged experiment passes your Euler function to the adapter's cached `denoise_step` path and compares it with the pinned model's `sample_actions`. Both use the same fixed noise, evaluation mode, and configured number of steps under `torch.no_grad()`. Inference caches the image, language, and state prefix once. The training helper `ADAPTER.velocity` recomputes the full prefix with the action suffix on each call, so it is a different numeric path and is not the reference for exact sampler parity.

In [ ]:
assert torch.allclose(
    flow_training_pair(torch.tensor([[[2.0]]]), torch.tensor([[[-1.0]]]), torch.tensor([0.25]))[0],
    torch.tensor([[[1.25]]]),
)
assert (
    flow_training_pair(torch.tensor([[[2.0]]]), torch.tensor([[[-1.0]]]), torch.tensor([0.25]))[
        1
    ].item()
    == -3.0
)
assert (
    masked_flow_mse(
        torch.ones(1, 2, 3), torch.zeros(1, 2, 3), torch.ones(1, 2, dtype=torch.bool), 2
    ).item()
    == 0.0
)
print("flow pair, padding mask, and Euler direction checks passed")

# More behaviour checks: lesson19_checks.py, next to this notebook.
from lesson19_checks import check_euler_to_action, check_masked_flow_mse

check_masked_flow_mse(masked_flow_mse)
check_euler_to_action(euler_to_action)

**Explain (2–3 sentences).** Why is $\Delta t$ negative? Which dimensions are excluded from the loss and why?

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
POLICY, PREPROCESSOR, POSTPROCESSOR = load_prepared_smolvla("published", DATA, device=DEVICE)
POLICY.eval()
ACTION_CHUNK_SIZE = POLICY.config.chunk_size
DATASET = open_libero_dataset(
    CONFIGS,
    DATA,
    "bowl_plate",
    "validation",
    {"action": frame_offset_seconds(list(range(ACTION_CHUNK_SIZE)))},
)
RAW = default_collate([DATASET[0]])
RAW["task"] = ["put the bowl on the plate"]
BATCH = PREPROCESSOR(RAW)
BATCH = {
    key: value.to(DEVICE) if isinstance(value, torch.Tensor) else value
    for key, value in BATCH.items()
}
ADAPTER = SmolVLAAdapter(POLICY)
INPUTS = ADAPTER.prepare_inputs(BATCH)
INITIAL_NOISE = torch.randn(1, ADAPTER.chunk_size, ADAPTER.max_action_dim, device=DEVICE)
NUM_STEPS = POLICY.config.num_steps
with torch.no_grad():
    OURS = ADAPTER.sample_chunk(
        INPUTS, sampler=euler_to_action, noise=INITIAL_NOISE, num_steps=NUM_STEPS
    )
    PINNED_PADDED = POLICY.model.sample_actions(
        INPUTS["images"],
        INPUTS["img_masks"],
        INPUTS["lang_tokens"],
        INPUTS["lang_masks"],
        INPUTS["state"],
        noise=INITIAL_NOISE,
    )
PINNED = PINNED_PADDED[:, :, : ADAPTER.action_dim]
assert torch.equal(OURS, PINNED), (OURS - PINNED).abs().max().item()
print("exact sampler match; steps:", NUM_STEPS, "| output shape:", tuple(PINNED.shape))

Both paths use the same model, observation, initial noise, and number of steps. This checks the sampler implementation. It does not compare task success.